In [1]:
import ast
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr

In [2]:
CSV_PATH = "C:\\Users\\hp\\Downloads\\streaming_content_trends.csv"

In [3]:
df = pd.read_csv(CSV_PATH)

df["genres"] = df["genres"].fillna("Unknown")
df["origin_country"] = df["origin_country"].fillna("Unknown")
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")


In [4]:
LANG_NAMES = {
    "en": "English", "ja": "Japanese", "ko": "Korean", "zh": "Chinese",
    "es": "Spanish", "fr": "French", "de": "German", "ru": "Russian",
    "ta": "Tamil", "hi": "Hindi", "it": "Italian", "pt": "Portuguese",
    "th": "Thai", "tr": "Turkish", "sv": "Swedish", "da": "Danish",
    "nl": "Dutch", "pl": "Polish", "no": "Norwegian", "cs": "Czech",
}
df["language"] = df["original_language"].map(LANG_NAMES).fillna(df["original_language"])

In [5]:
genre_rows = []
for _, row in df.iterrows():
    for g in [g.strip() for g in str(row["genres"]).split(",") if g.strip()]:
        genre_rows.append({"genre": g, "media_type": row["media_type"],
                            "popularity": row["popularity"], "vote_average": row["vote_average"],
                            "search_category": row["search_category"]})
genre_df = pd.DataFrame(genre_rows)

ALL_GENRES = sorted(genre_df["genre"].unique().tolist())
ALL_CATEGORIES = sorted(df["search_category"].unique().tolist())
ALL_MEDIA_TYPES = sorted(df["media_type"].unique().tolist())
ALL_LANGUAGES = sorted(df["language"].dropna().unique().tolist())

MIN_YEAR = int(df["release_year"].min(skipna=True))
MAX_YEAR = int(df["release_year"].max(skipna=True))

TEMPLATE = "plotly_white"

In [6]:
def filter_df(media_types, categories, genres, languages, year_lo, year_hi, min_votes):
    d = df.copy()
    if media_types:
        d = d[d["media_type"].isin(media_types)]
    if categories:
        d = d[d["search_category"].isin(categories)]
    if languages:
        d = d[d["language"].isin(languages)]
    if genres:
        pattern = "|".join([g.replace("(", r"\(").replace(")", r"\)") for g in genres])
        d = d[d["genres"].str.contains(pattern, case=False, na=False)]
    if year_lo is not None and year_hi is not None:
        lo, hi = min(year_lo, year_hi), max(year_lo, year_hi)
        d = d[(d["release_year"].isna()) | ((d["release_year"] >= lo) & (d["release_year"] <= hi))]
    if min_votes:
        d = d[d["vote_count"] >= min_votes]
    return d

In [7]:
def kpi_cards(d):
    n = len(d)
    avg_pop = d["popularity"].mean() if n else 0
    avg_rating = d["vote_average"].mean() if n else 0
    n_movie = int((d["media_type"] == "movie").sum())
    n_tv = int((d["media_type"] == "tv").sum())
    return (
        f"### 🎬 {n:,}\nTitles matching filters",
        f"### ⭐ {avg_rating:.2f}\nAverage rating",
        f"### 🔥 {avg_pop:,.0f}\nAverage popularity",
        f"### 📺 {n_movie:,} movies / {n_tv:,} TV",
    )

In [8]:
def fig_media_split(d):
    counts = d["media_type"].value_counts().reset_index()
    counts.columns = ["media_type", "count"]
    fig = px.pie(counts, names="media_type", values="count", hole=0.55,
                 title="Movies vs TV Shows", template=TEMPLATE,
                 color_discrete_sequence=px.colors.qualitative.Set2)
    fig.update_traces(textinfo="percent+label")
    return fig

In [9]:
def fig_category_split(d):
    counts = d["search_category"].value_counts().reset_index()
    counts.columns = ["search_category", "count"]
    fig = px.bar(counts, x="search_category", y="count", title="Titles by Search Category",
                 template=TEMPLATE, color="search_category",
                 color_discrete_sequence=px.colors.qualitative.Set2, text="count")
    fig.update_layout(showlegend=False)
    return fig

In [10]:
def fig_top_genres(d):
    # build genre rows restricted to the currently-filtered titles
    sub = d[["id", "genres", "popularity"]].copy()
    rows = []
    for _, r in sub.iterrows():
        for g in [g.strip() for g in str(r["genres"]).split(",") if g.strip()]:
            rows.append({"genre": g, "popularity": r["popularity"]})
    gsub = pd.DataFrame(rows)
    if gsub.empty:
        return go.Figure().update_layout(title="No data for current filters", template=TEMPLATE)
    top = (gsub.groupby("genre").agg(count=("genre", "size"), avg_pop=("popularity", "mean"))
           .reset_index().sort_values("count", ascending=False).head(15))
    fig = px.bar(top.sort_values("count"), x="count", y="genre", orientation="h",
                 title="Top 15 Genres by Title Count", template=TEMPLATE,
                 color="avg_pop", color_continuous_scale="Viridis",
                 labels={"avg_pop": "Avg Popularity", "count": "Titles"})
    return fig

In [11]:
def fig_popularity_vs_rating(d):
    if d.empty:
        return go.Figure().update_layout(title="No data for current filters", template=TEMPLATE)
    fig = px.scatter(d, x="vote_average", y="popularity", color="media_type",
                      size="vote_count", hover_name="title", template=TEMPLATE,
                      title="Popularity vs. Rating (bubble size = vote count)",
                      color_discrete_sequence=px.colors.qualitative.Set2,
                      opacity=0.7, log_y=True)
    return fig

In [13]:
def fig_release_trend(d):
    yd = d.dropna(subset=["release_year"])
    if yd.empty:
        return go.Figure().update_layout(title="No data for current filters", template=TEMPLATE)
    yd = yd[(yd["release_year"] >= 1990)]
    counts = yd.groupby(["release_year", "media_type"]).size().reset_index(name="count")
    fig = px.line(counts, x="release_year", y="count", color="media_type", markers=True,
                  title="Titles by Release Year (1990+)", template=TEMPLATE,
                  color_discrete_sequence=px.colors.qualitative.Set2)
    return fig

In [14]:
def fig_language(d):
    counts = d["language"].value_counts().head(10).reset_index()
    counts.columns = ["language", "count"]
    fig = px.bar(counts.sort_values("count"), x="count", y="language", orientation="h",
                 title="Top 10 Original Languages", template=TEMPLATE,
                 color="count", color_continuous_scale="Blues")
    fig.update_layout(coloraxis_showscale=False)
    return fig

In [15]:
def top_titles_table(d, n=15):
    cols = ["title", "media_type", "search_category", "genres", "language",
            "release_year", "popularity", "vote_average", "vote_count"]
    return (d.sort_values("popularity", ascending=False)[cols]
            .head(n).round({"popularity": 1, "vote_average": 2}))


In [16]:
def update_dashboard(media_types, categories, genres, languages, year_lo, year_hi, min_votes):
    d = filter_df(media_types, categories, genres, languages, year_lo, year_hi, min_votes)
    k1, k2, k3, k4 = kpi_cards(d)
    return (
        k1, k2, k3, k4,
        fig_media_split(d), fig_category_split(d),
        fig_top_genres(d), fig_popularity_vs_rating(d),
        fig_release_trend(d), fig_language(d),
        top_titles_table(d),
    )

In [17]:
with gr.Blocks(title="Streaming Content Trends Dashboard", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 Streaming Content Trends Dashboard")
    gr.Markdown("Explore popular, top-rated, and trending movies & TV shows. Use the filters on the left to slice the data.")

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            gr.Markdown("### Filters")
            media_type_filter = gr.CheckboxGroup(choices=ALL_MEDIA_TYPES, value=ALL_MEDIA_TYPES,
                                                   label="Media Type")
            category_filter = gr.CheckboxGroup(choices=ALL_CATEGORIES, value=ALL_CATEGORIES,
                                                label="Search Category")
            genre_filter = gr.Dropdown(choices=ALL_GENRES, value=[], multiselect=True,
                                        label="Genre (any match)")
            language_filter = gr.Dropdown(choices=ALL_LANGUAGES, value=[], multiselect=True,
                                           label="Original Language")
            year_lo_filter = gr.Slider(minimum=MIN_YEAR, maximum=MAX_YEAR,
                                        value=max(MIN_YEAR, 1990), step=1,
                                        label="Release Year: From")
            year_hi_filter = gr.Slider(minimum=MIN_YEAR, maximum=MAX_YEAR,
                                        value=MAX_YEAR, step=1,
                                        label="Release Year: To")
            votes_filter = gr.Slider(minimum=0, maximum=int(df["vote_count"].quantile(0.95)),
                                      value=0, step=50, label="Minimum Vote Count")
            reset_btn = gr.Button("Reset Filters")

        with gr.Column(scale=3):
            with gr.Row():
                kpi1 = gr.Markdown()
                kpi2 = gr.Markdown()
                kpi3 = gr.Markdown()
                kpi4 = gr.Markdown()

            with gr.Tab("Overview"):
                with gr.Row():
                    plot_media_split = gr.Plot()
                    plot_category_split = gr.Plot()
                plot_genres = gr.Plot()

            with gr.Tab("Popularity & Ratings"):
                plot_scatter = gr.Plot()
                with gr.Row():
                    plot_trend = gr.Plot()
                    plot_language = gr.Plot()

            with gr.Tab("Top Titles"):
                table = gr.Dataframe(label="Top titles by popularity (current filters)",
                                      wrap=True)

    inputs = [media_type_filter, category_filter, genre_filter, language_filter,
              year_lo_filter, year_hi_filter, votes_filter]
    outputs = [kpi1, kpi2, kpi3, kpi4, plot_media_split, plot_category_split,
               plot_genres, plot_scatter, plot_trend, plot_language, table]

    for inp in inputs:
        inp.change(update_dashboard, inputs=inputs, outputs=outputs)

    def reset():
        return ALL_MEDIA_TYPES, ALL_CATEGORIES, [], [], max(MIN_YEAR, 1990), MAX_YEAR, 0

    reset_btn.click(reset, outputs=inputs).then(update_dashboard, inputs=inputs, outputs=outputs)

    demo.load(update_dashboard, inputs=inputs, outputs=outputs)


if __name__ == "__main__":
    demo.launch()

C:\Users\hp\AppData\Local\Temp\ipykernel_15268\4181551944.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Streaming Content Trends Dashboard", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
